# Passo 2 — Metodologia de avaliação

Sobre o artefato do Passo 1 (`prepared_measured_k4.npz`), com normalização do Barino mantida:

1. **Hold-out final** (20%) — só para relatório; não entra em ajuste.
2. **Estratégia A** — `StratifiedKFold` (5 folds) no conjunto de desenvolvimento.
3. **Estratégia B** — `RepeatedStratifiedKFold` (5 folds × 5 repetições) no mesmo desenvolvimento.

Estratificação: **quantis de** $\lambda_{res}$ (10 bins). Multi-rótulo não admite estratificação direta em 13 bits; bins de $\lambda_{res}$ são a chave física documentada no guia.

SMOTE **não** é aplicado aqui — só a metodologia de split.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.cv_utils import SplitConfig, build_and_save_splits, load_cv_splits
from src.data_utils import FIGURES_DIR, RESULTS_DIR, load_prepared_dataset

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

config = SplitConfig()
print(config)

## 1. Carregar dados preparados

In [ ]:
data = load_prepared_dataset()
X, y_mask, target = data["X"], data["y_mask"], data["target"]
print(f"X={X.shape}, y_mask={y_mask.shape}")
print(f"λ_res: [{target.min():.4f}, {target.max():.4f}] nm")
print(f"Máscaras únicas: {len({tuple(r.tolist()) for r in y_mask})}")

## 2. Construir hold-out + estratégias A e B

In [ ]:
out = build_and_save_splits(target, y_mask, config=config)
meta = out["meta"]
print(json.dumps(meta, indent=2, ensure_ascii=False))

## 3. Coerência do hold-out

In [ ]:
dev_idx, holdout_idx = out["dev_idx"], out["holdout_idx"]
assert len(np.intersect1d(dev_idx, holdout_idx)) == 0
assert len(dev_idx) + len(holdout_idx) == len(target)

bal = pd.DataFrame(
    {
        "fbg": np.arange(13),
        "frac_global": np.round(y_mask.mean(axis=0), 4),
        "frac_dev": np.round(y_mask[dev_idx].mean(axis=0), 4),
        "frac_holdout": np.round(y_mask[holdout_idx].mean(axis=0), 4),
    }
)
bal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(target[dev_idx], bins=30, color="#4c72b0", edgecolor="white", alpha=0.9, label="dev")
axes[0].hist(target[holdout_idx], bins=30, color="#c44e52", edgecolor="white", alpha=0.7, label="hold-out")
axes[0].set_xlabel(r"$\lambda_{res}$ (nm)")
axes[0].set_ylabel("Contagem")
axes[0].set_title("Dev vs hold-out")
axes[0].legend(fontsize=8)

bins = out["lambda_bins"]
all_b = np.unique(bins)
dev_frac = [(bins[dev_idx] == b).mean() for b in all_b]
ho_frac = [(bins[holdout_idx] == b).mean() for b in all_b]
x = np.arange(len(all_b))
w = 0.4
axes[1].bar(x - w / 2, dev_frac, width=w, label="dev", color="#4c72b0")
axes[1].bar(x + w / 2, ho_frac, width=w, label="hold-out", color="#c44e52")
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(int(b)) for b in all_b])
axes[1].set_xlabel("Bin (quantil)")
axes[1].set_ylabel("Fração")
axes[1].set_title("Proporção por bin")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "passo2_holdout_vs_dev.png", dpi=150)
plt.show()

## 4. Estratégia A — StratifiedKFold

In [ ]:
sa = out["summary_a"]
print("min positivos no teste (qualquer FBG):", meta["strategy_a"]["min_test_pos_any_fbg"])
sa

## 5. Estratégia B — RepeatedStratifiedKFold

In [ ]:
sb = out["summary_b"]
print(f"n_evaluations = {len(sb)}")
print("min positivos no teste (qualquer FBG):", meta["strategy_b"]["min_test_pos_any_fbg"])
print(
    "std(lambda_mean_test): "
    f"A={sa['lambda_mean_test'].std(ddof=0):.4f}, "
    f"B={sb['lambda_mean_test'].std(ddof=0):.4f}"
)
print("FBG12 (raro) no teste — A:", sa["test_pos_fbg12"].tolist())
print(
    "FBG12 no teste — B min/median/max:",
    int(sb["test_pos_fbg12"].min()),
    float(sb["test_pos_fbg12"].median()),
    int(sb["test_pos_fbg12"].max()),
)
sb.head(10)

## 6. Comparação objetiva (para escolher depois)

| Critério | A | B |
|----------|---|---|
| Custo computacional | 5 avaliações | 25 avaliações |
| Estabilidade da estimativa | uma passagem | média ± desvio entre repetições |
| Mesma estratificação | sim ($\lambda_{res}$ bins) | sim |
| Hold-out isolado | sim | sim |

Artefato: `results/cv_splits_passo2.npz`. Escolher **uma** estratégia antes do Passo 3.

In [ ]:
# Reload I/O
reloaded = load_cv_splits()
assert len(reloaded["dev_idx"]) == meta["n_dev"]
assert len(reloaded["holdout_idx"]) == meta["n_holdout"]
assert len(reloaded["a_train"]) == config.n_splits
assert len(reloaded["b_train"]) == config.n_splits * config.n_repeats
print("Reload OK")